# Load top history solution and materialize runnable repo
Select from history using the same logic as optimization, then write a runnable workspace/repo with that selected code.

In [ ]:
from pathlib import Path
import ast
import json
import shutil
import re
import textwrap
from typing import Any
from tusoai.optimization import (
    _dm_collect_function_sources,
    _dm_extract_base_functions,
    _dm_init_repo_snapshots,
    _dm_prepare_eval_workspace,
    _dm_apply_function_updates,
    _dm_load_history_records_pool,
    _dm_read_history_entries,
    _dm_get_selected_history_summary,
    _dm_history_close_set,
    _dm_history_complexity_score,
    _dm_target_def_name,
    _find_def_header_colon,
    _DMRunState,
    _DMPrinter,
    MethodTask,
    DataTask,
)


In [ ]:
def _function_header_span(code: str, function_name: str):
    """Return (start, end) for the target function header, including multiline signatures."""
    target_def_name = _dm_target_def_name(function_name)
    pattern = re.compile(rf"(?m)^[ \t]*(?:async\s+)?def[ \t]+{re.escape(target_def_name)}[ \t]*\(")
    match = pattern.search(code or '')
    if not match:
        return None
    colon_pos = _find_def_header_colon(code, match.end() - 1)
    if colon_pos is None:
        return None
    return match.start(), colon_pos + 1


def _dedented_header(code: str, function_name: str) -> str | None:
    span = _function_header_span(code, function_name)
    if span is None:
        return None
    return textwrap.dedent(code[span[0]:span[1]]).strip('\n')


def _align_history_function_signature(fn_code: str, baseline_code: str, function_name: str) -> tuple[str, bool]:
    """
    Preserve the current source signature when materializing history code.

    Older histories can contain a selected body with a now-stale/generated signature.  The
    rebuilt repo should keep the same callable interface as the current target source while
    still using the selected implementation body.
    """
    baseline_header = _dedented_header(baseline_code, function_name)
    selected_span = _function_header_span(fn_code, function_name)
    if not baseline_header or selected_span is None:
        return fn_code, False

    selected_header = textwrap.dedent(fn_code[selected_span[0]:selected_span[1]]).strip('\n')
    if selected_header == baseline_header:
        return fn_code, False

    selected_line = fn_code[selected_span[0]:].splitlines()[0]
    selected_indent = selected_line[: len(selected_line) - len(selected_line.lstrip(' \t'))]
    replacement_header = textwrap.indent(baseline_header, selected_indent)
    return fn_code[:selected_span[0]] + replacement_header + fn_code[selected_span[1]:], True


def _print_selected_history_code(selected_record, ordered_fn_names):
    print('Selected top-performing history code:')
    print(f"accuracy={float(selected_record.accuracy):.6f} runtime={float(selected_record.runtime):.3f}s lineage={selected_record.lineage}")
    for fn in ordered_fn_names:
        print(f"\n--- {fn} ---")
        print(selected_record.functions.get(fn, selected_record.code))



def _read_json_if_exists(path: Path):
    """Read JSON logs, including compact tusoai.history.v2 files."""
    if not path.exists() or not path.is_file():
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(data, dict) and data.get('format') == 'tusoai.history.v2':
        return {'history': _dm_read_history_entries(path), 'source_path': str(path)}
    return data


def _walk_dicts(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from _walk_dicts(value)
    elif isinstance(obj, list):
        for item in obj:
            yield from _walk_dicts(item)


def _history_sidecar_candidates(history_path: str | Path) -> list[Path]:
    """Return likely sidecar logs that contain optimization run wiring metadata."""
    hpath = Path(history_path).expanduser()
    hdir = hpath.parent if hpath.suffix == '.json' else hpath

    candidates: list[Path] = []

    def add(path: Path):
        if path not in candidates:
            candidates.append(path)

    # Canonical TusoAI run layout.
    for name in ('dev.json', 'prompt_io.json', 'history.json'):
        add(hdir / name)

    # Some users copy/rename sidecars. Prefer local files with descriptive names.
    if hdir.exists():
        for pattern in ('*dev*.json', '*prompt*io*.json', '*prompt*.json', '*history*.json', '*.json'):
            for path in sorted(hdir.glob(pattern)):
                add(path)

    # If the history file is nested one level below the run folder, also check the parent.
    parent = hdir.parent
    if parent != hdir and parent.exists():
        for name in ('dev.json', 'prompt_io.json'):
            add(parent / name)

    if hpath.is_file():
        add(hpath)
    return candidates


def _valid_inferred_config(d: dict[str, Any], source_log: Path) -> dict[str, Any] | None:
    """Return normalized run wiring if a dictionary contains enough metadata."""
    if 'function_sources' not in d:
        return None
    function_sources = d.get('function_sources') or {}
    if not isinstance(function_sources, dict) or not function_sources:
        return None

    ordered_fn_names = list(d.get('ordered_fn_names') or function_sources.keys())
    if not ordered_fn_names or not all(fn in function_sources for fn in ordered_fn_names):
        return None

    reference_filename = d.get('reference_filename')
    if reference_filename is None:
        # Usually all inline targets share the runner/reference file.
        first_source = function_sources.get(ordered_fn_names[0], {})
        reference_filename = first_source.get('file_path') or first_source.get('source_path')
    if reference_filename is None:
        return None

    return {
        'reference_filename': str(reference_filename),
        'ordered_fn_names': ordered_fn_names,
        'function_sources': function_sources,
        'source_log': str(source_log),
    }


def _infer_ordered_fn_names_from_history(history_path: str | Path) -> list[str]:
    """Infer target function labels from ### sections in history code blocks."""
    names: list[str] = []
    for entry in _dm_read_history_entries(history_path):
        code = entry.get('code', '') if isinstance(entry, dict) else ''
        for match in re.finditer(r'(?m)^###\s+(.+?)\s*$', code or ''):
            name = match.group(1).strip()
            if name and name not in names:
                names.append(name)
        if names:
            return names
    return names


def infer_history_run_config(history_path: str | Path) -> dict[str, Any]:
    """
    Infer reference_filename, ordered_fn_names, and function_sources from run logs.

    The preferred source is sidecar metadata written by recent TusoAI versions.  The search is
    intentionally tolerant of renamed/copied dev and prompt I/O files and compact history files.
    """
    best = None
    partial_with_sources = None
    for sidecar in _history_sidecar_candidates(history_path):
        data = _read_json_if_exists(sidecar)
        if data is None:
            continue
        for d in _walk_dicts(data):
            config = _valid_inferred_config(d, sidecar)
            if config is not None:
                best = config
            elif isinstance(d.get('function_sources'), dict) and d.get('function_sources'):
                partial_with_sources = (sidecar, d)

    if best is not None:
        return best

    # Last-chance recovery for logs that have function_sources but not ordered_fn_names.
    if partial_with_sources is not None:
        sidecar, d = partial_with_sources
        ordered_fn_names = _infer_ordered_fn_names_from_history(history_path) or list(d['function_sources'].keys())
        d = dict(d)
        d['ordered_fn_names'] = [fn for fn in ordered_fn_names if fn in d['function_sources']]
        config = _valid_inferred_config(d, sidecar)
        if config is not None:
            return config

    inferred_names = _infer_ordered_fn_names_from_history(history_path)
    if inferred_names:
        return {
            'reference_filename': None,
            'ordered_fn_names': inferred_names,
            'function_sources': {},
            'source_log': str(history_path),
            'history_only': True,
            'checked_sidecars': [str(p) for p in _history_sidecar_candidates(history_path)],
        }

    checked = ', '.join(str(p) for p in _history_sidecar_candidates(history_path))
    raise ValueError(
        'Could not infer run wiring from the history path. Expected nearby dev/prompt_io logs '
        'containing reference_filename and function_sources, or history code blocks with ### target names. '
        f'Checked: {checked}'
    )


def _safe_history_filename(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', name).strip('_') or 'selected_function'


def _write_history_only_solution(
    *,
    selected_record,
    ordered_fn_names: list[str],
    output_path: Path,
    picked: dict[str, Any],
    inferred_config: dict[str, Any],
) -> dict[str, Any]:
    """Write selected history code when source-path sidecars are unavailable."""
    selected_dir = output_path / 'selected_code'
    selected_dir.mkdir(parents=True, exist_ok=True)

    bundle_path = selected_dir / 'selected_history_code.py'
    bundle_path.write_text(selected_record.code.rstrip() + '\n', encoding='utf-8')

    function_files = {}
    for fn in ordered_fn_names:
        fn_code = selected_record.functions.get(fn, selected_record.code).rstrip() + '\n'
        fn_path = selected_dir / f'{_safe_history_filename(fn)}.py'
        fn_path.write_text(fn_code, encoding='utf-8')
        function_files[fn] = str(fn_path)

    metadata_path = selected_dir / 'selected_summary.json'
    metadata = {
        'selected': {k: v for k, v in picked.items() if k != 'record'},
        'ordered_fn_names': ordered_fn_names,
        'bundle_file': str(bundle_path),
        'function_files': function_files,
        'inferred_config_source': inferred_config.get('source_log'),
        'checked_sidecars': inferred_config.get('checked_sidecars', []),
        'history_only': True,
        'note': (
            'No dev/prompt_io sidecar with function_sources was found, so the notebook wrote '
            'the selected history code instead of patching a runnable source tree. Pass '
            'reference_filename plus method_tasks/data_tasks, or place a sidecar with '
            'function_sources next to history.json, to materialize a runnable repo.'
        ),
    }
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

    return {
        'selected': metadata['selected'],
        'signature_aligned': [],
        'workspace': str(selected_dir),
        'runner_file': str(bundle_path),
        'repo_workspaces': {},
        'ordered_fn_names': ordered_fn_names,
        'inferred_config_source': inferred_config.get('source_log'),
        'history_only': True,
        'function_files': function_files,
        'metadata_file': str(metadata_path),
        'message': metadata['note'],
    }


def _normalize_function_sources(function_sources: dict[str, dict[str, Any]]) -> dict[str, dict[str, Any]]:
    """Convert serialized paths to the shapes expected by optimization workspace helpers."""
    normalized = {}
    for fn, source in function_sources.items():
        item = dict(source)
        for key in ('file_path', 'repo_root', 'package_copy_dir'):
            if item.get(key) is not None:
                item[key] = str(item[key])
        normalized[fn] = item
    return normalized

def pick_history_solution(history_path: str, ordered_fn_names, min_improvement: float = 0.001):
    """Pick using optimization-identical selection logic."""
    records = _dm_load_history_records_pool(history_path, ordered_fn_names)
    if not records:
        raise ValueError('No valid history entries that contain all requested functions.')

    close = _dm_history_close_set(records, min_improvement=min_improvement)
    top_accuracy = max(close, key=lambda m: float(m.accuracy))
    selected = max(close, key=lambda m: _dm_history_complexity_score(m, top_accuracy))

    summary = _dm_get_selected_history_summary(history_path, accuracy_tolerance=min_improvement) or {}
    return {
        'record': selected,
        'history_count': summary.get('history_count', len(records)),
        'best_accuracy': summary.get('best_accuracy', max(float(r.accuracy) for r in records)),
        'accuracy': float(selected.accuracy),
        'runtime': float(selected.runtime),
        'code_len_lines': int(len([ln for ln in selected.code.splitlines() if ln.strip()])),
        'lineage': selected.lineage,
        'selection_score': float(_dm_history_complexity_score(selected, top_accuracy)),
    }

def build_selected_history_solution(
    *,
    history_path: str,
    output_dir: str = 'history_rebuild_output',
    reference_filename: str | None = None,
    method_tasks: list | None = None,
    data_tasks: list | None = None,
    min_improvement: float = 0.001,
    print_selected_code: bool = True,
):
    """
    Recreate the same workspace/repo wiring used by optimize(), apply selected history functions,
    and write the runnable build into output_dir.

    By default, run wiring is inferred from metadata logs next to history.json. The old
    explicit reference_filename/method_tasks/data_tasks arguments are still accepted as a
    fallback for older histories without sidecar metadata.
    """
    data_tasks = data_tasks or []
    output_path = Path(output_dir)
    if output_path.exists():
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    inferred_config = None
    if reference_filename is None or method_tasks is None:
        inferred_config = infer_history_run_config(history_path)
        ordered_fn_names = list(inferred_config['ordered_fn_names'])
        picked = pick_history_solution(history_path, ordered_fn_names, min_improvement=min_improvement)
        selected = picked['record']

        if print_selected_code:
            _print_selected_history_code(selected, ordered_fn_names)

        if inferred_config.get('history_only'):
            return _write_history_only_solution(
                selected_record=selected,
                ordered_fn_names=ordered_fn_names,
                output_path=output_path,
                picked=picked,
                inferred_config=inferred_config,
            )

        reference_filename = reference_filename or inferred_config['reference_filename']
        function_sources = _normalize_function_sources(inferred_config['function_sources'])
    else:
        function_sources = _dm_collect_function_sources(method_tasks, data_tasks, reference_filename)
        ordered_fn_names = list(function_sources.keys())
        picked = pick_history_solution(history_path, ordered_fn_names, min_improvement=min_improvement)
        selected = picked['record']

        if print_selected_code:
            _print_selected_history_code(selected, ordered_fn_names)

    base_functions = _dm_extract_base_functions(ordered_fn_names, function_sources)
    selected_functions = dict(selected.functions)
    signature_aligned = []
    for fn in ordered_fn_names:
        aligned_code, changed = _align_history_function_signature(selected_functions[fn], base_functions[fn], fn)
        selected_functions[fn] = aligned_code
        if changed:
            signature_aligned.append(fn)

    if signature_aligned:
        print('Aligned selected history function signature(s) to current source:', ', '.join(signature_aligned))

    base_path = output_path / 'build_workspace'
    base_path.mkdir(parents=True, exist_ok=True)
    repo_snaps = _dm_init_repo_snapshots(function_sources, base_path)
    eval_path, repo_workspaces = _dm_prepare_eval_workspace(
        base_path=base_path,
        reference_filename=reference_filename,
        repo_snapshots=repo_snaps,
        function_sources=function_sources,
        safe_tag='history_selected',
    )

    _dm_apply_function_updates(
        functions=selected_functions,
        ordered_fn_names=ordered_fn_names,
        function_sources=function_sources,
        eval_path=eval_path,
        repo_workspaces=repo_workspaces,
        capture_function_prints=False,
    )

    return {
        'selected': {k: v for k, v in picked.items() if k != 'record'},
        'signature_aligned': signature_aligned,
        'workspace': str(base_path),
        'runner_file': str(eval_path),
        'repo_workspaces': {k: str(v) for k, v in repo_workspaces.items()},
        'ordered_fn_names': ordered_fn_names,
        'inferred_config_source': inferred_config.get('source_log') if inferred_config else None,
    }


In [ ]:
# Input only your history file location. The notebook infers the runner and source wiring from adjacent run logs.
history_path = 'path/to/history.json'

out = build_selected_history_solution(
    history_path=history_path,
    output_dir='history_rebuild_output',
    min_improvement=0.001,
)
print(out['selected'])
print('inferred_config_source:', out['inferred_config_source'])
print('runner_file:', out['runner_file'])
print('repo_workspaces:', out['repo_workspaces'])


In [ ]:
# Plot top performance over iterations and cost, and flag significant improvements
import math
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt


def _history_cost_value(entry: dict, previous_total: float) -> tuple[float, float]:
    """Return (cumulative_cost, incremental_cost) for one history entry.

    Newer histories record cumulative ``total_cost``; older/dev entries may only
    record per-entry ``cost`` or ``local_cost``. Prefer the cumulative value when
    present so the secondary cost axis does not stay at $0.00 for optimization
    entries whose prompt cost has already been rolled up into ``total_cost``.
    """
    for key in ('total_cost', 'cum_cost', 'cumulative_cost'):
        value = entry.get(key)
        if value is None:
            continue
        try:
            total = float(value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(total):
            return total, max(0.0, total - previous_total)

    incremental = 0.0
    for key in ('cost', 'local_cost'):
        value = entry.get(key)
        if value is None:
            continue
        try:
            amount = float(value or 0.0)
        except (TypeError, ValueError):
            amount = 0.0
        if math.isfinite(amount):
            incremental += amount
    return previous_total + incremental, incremental


def _improvement_label(index: int) -> str:
    """One-based numeric labels for significant improvements."""
    return str(index + 1)


def _bbox_overlaps(candidate, placed_bboxes) -> bool:
    """Return True when an annotation bbox overlaps an existing label bbox."""
    return any(candidate.overlaps(existing) for existing in placed_bboxes)


def plot_history_progress(history_path: str, sig_improvement: float = 0.01):
    data = json.loads(Path(history_path).read_text(encoding='utf-8'))
    entries = data if isinstance(data, list) else data.get('history', [])

    points = []
    cum_cost = 0.0
    for _i, e in enumerate(entries):
        if not isinstance(e, dict):
            continue
        try:
            acc = float(e.get('accuracy'))
        except Exception:
            continue
        if not math.isfinite(acc):
            continue
        cum_cost, step_cost = _history_cost_value(e, cum_cost)
        points.append({
            'iter': len(points),
            'acc': acc,
            'cum_cost': cum_cost,
            'cost': step_cost,
            'code': e.get('code', ''),
            'lineage': e.get('lineage', ''),
        })

    if not points:
        raise ValueError('No valid history entries with numeric accuracy.')

    best_so_far = []
    curr_best = float('-inf')
    sig_hits = []
    significant_iters = set()
    for p in points:
        if p['acc'] > curr_best:
            gain = p['acc'] - curr_best if curr_best != float('-inf') else 0.0
            if curr_best != float('-inf') and gain >= sig_improvement:
                sig_hits.append((p, gain))
                significant_iters.add(p['iter'])
            curr_best = p['acc']
        best_so_far.append(curr_best)

    plt.rcParams.update({
        'axes.titlesize': 18,
        'axes.labelsize': 15,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
    })
    fig, ax1 = plt.subplots(figsize=(8, 8))
    xs = [p['iter'] for p in points]
    nonsig = [p for p in points if p['iter'] not in significant_iters]

    if nonsig:
        ax1.scatter(
            [p['iter'] for p in nonsig],
            [p['acc'] for p in nonsig],
            color='tab:red',
            alpha=0.3,
            s=52,
            edgecolors='none',
            label='Attempt',
            zorder=2,
        )
    ax1.plot(xs, best_so_far, color='tab:blue', linewidth=2.8, label='Best so far', zorder=3)
    ax1.set_ylim(bottom=points[0]['acc'])

    # Mark significant improvements with outlined green dots and outlined numeric labels.
    # Labels are moved upward until their rendered bounding boxes no longer overlap.
    label_effect = [pe.withStroke(linewidth=3.5, foreground='black')]
    placed_label_bboxes = []
    for hit_idx, (p, _gain) in enumerate(sig_hits):
        ax1.scatter(
            [p['iter']],
            [p['acc']],
            facecolors='limegreen',
            edgecolors='black',
            linewidths=1.6,
            s=115,
            zorder=5,
        )
        label = ax1.annotate(
            _improvement_label(hit_idx),
            (p['iter'], p['acc']),
            xytext=(7, 8),
            textcoords='offset points',
            color='limegreen',
            fontsize=15,
            fontweight='bold',
            path_effects=label_effect,
            zorder=6,
        )
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        label_bbox = label.get_window_extent(renderer).expanded(1.15, 1.25)
        offset_y = 8
        while _bbox_overlaps(label_bbox, placed_label_bboxes):
            offset_y += 14
            label.set_position((7, offset_y))
            fig.canvas.draw()
            label_bbox = label.get_window_extent(renderer).expanded(1.15, 1.25)
        placed_label_bboxes.append(label_bbox)

    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Performance')
    ax1.grid(alpha=0.22, linewidth=0.8)
    ax1.set_title(f'Performance progress (min gain {sig_improvement:g})', pad=14)
    ax1.legend(loc='best', frameon=True)

    ax2 = ax1.twiny()
    ax2.set_xlim(ax1.get_xlim())
    ticks = ax1.get_xticks()
    tick_iters = sorted({int(round(t)) for t in ticks if 0 <= int(round(t)) < len(points)})
    if tick_iters:
        ax2.set_xticks(tick_iters)
        ax2.set_xticklabels([f"${points[t]['cum_cost']:.2f}" for t in tick_iters])
    ax2.set_xlabel('Cumulative cost')

    fig.tight_layout()
    plt.show()

    if sig_hits:
        print(f'Significant improvements (>= {sig_improvement:.6f}):')
        for idx, (p, gain) in enumerate(sig_hits):
            print('---')
            print(f"{_improvement_label(idx)}: iter={p['iter']} gain={gain:.6f} acc={p['acc']:.6f} cost=${p['cum_cost']:.2f} lineage={p['lineage']}")
            print(p['code'])
    else:
        print(f'No significant improvements found for threshold {sig_improvement:.6f}.')

    return {'points': points, 'best_so_far': best_so_far, 'sig_hits': sig_hits}



In [ ]:
# Example plotting call:
plot_out = plot_history_progress(history_path, sig_improvement=0.01)
